<a href="https://colab.research.google.com/github/jessrat/AquaLimpia-tarea-semana-8/blob/main/notebooks/TareaSemana8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ==========================================
# AquaLimpia S.A.
# alumno  : Jorge Salas
# Fecha   : 25/05/2026
# Semana  : 8
# Carrera : Ingeniería en informática
# ==========================================

import pandas as pd
import numpy as np
import os

import plotly.express as px
from plotly.subplots import make_subplots

from joblib import dump


# ------------------------------------------
# 1. Cargar datos
# ------------------------------------------
def cargar_datos():
    from google.colab import files

    print ("\n Suba archivo .xlsx")
    uploaded = files.upload()
    df = pd.read_excel(list(uploaded.keys())[0])
    df.columns = df.columns.str.strip()
    return df


# ------------------------------------------
# 2. Evaluación de calidad de datos
# ------------------------------------------
def evaluar_calidad(df):
    print("\n ---- Validación de datos ----")

    print("\n--- Valores nulos ---")
    print(df.isnull().sum())

    print("\n--- Registros duplicados ---")
    print(df.duplicated().sum())

    print("\n--- Estadísticas generales ---")
    print(df.describe())


# ------------------------------------------
# 3. Limpieza de datos
# ------------------------------------------
def limpiar_datos(df):
    print("\n ---- Limpieza de datos ----")

    df = df.drop_duplicates()
    columnas_numericas = [
        "caudal_entrada_m3_d",
        "DBO_entrada_mg_L",
        "DBO_salida_mg_L",
        "energia_aeracion_kWh",
        "lodos_generados_kg_d"
    ]

    for col in columnas_numericas:
        df[col] = df[col].fillna(df[col].mean())

    return df

# ------------------------------------------
# 4. Guardar almacenamiento intermedio
# ------------------------------------------
def guardar_intermedio(df):

    os.makedirs("data/intermedio", exist_ok=True)

    ruta = "data/intermedio/datos_limpios.csv"
    df.to_csv(ruta, index=False)

    print(f" Datos intermedios guardados en: {ruta}")


# ------------------------------------------
# 5. Cálculo de indicadores
# ------------------------------------------
def calcular_indicadores(df):
    print("\n ---- Calculo indicadores ----")

    df["eficiencia_tratamiento"] = (
        (df["DBO_entrada_mg_L"] - df["DBO_salida_mg_L"]) /
        df["DBO_entrada_mg_L"]
    ) * 100

    df["energia_por_caudal"] = (
        df["energia_aeracion_kWh"] / df["caudal_entrada_m3_d"]
    )

    indicadores = {
        "promedio_eficiencia": np.mean(df["eficiencia_tratamiento"]),
        "max_eficiencia": np.max(df["eficiencia_tratamiento"]),
        "min_eficiencia": np.min(df["eficiencia_tratamiento"]),
        "promedio_energia": np.mean(df["energia_por_caudal"])
    }

    return df, indicadores




# ------------------------------------------
# 6. Agrupación por planta
# ------------------------------------------
def resumen_por_planta(df):
    resumen = df.groupby("planta").agg({
        "caudal_entrada_m3_d": "mean",
        "DBO_salida_mg_L": "mean",
        "eficiencia_tratamiento": "mean",
        "energia_aeracion_kWh": "mean",
        "lodos_generados_kg_d": "mean"
    }).reset_index()

    return resumen


# ------------------------------------------
# 7. Dashboard exploratorio
# ------------------------------------------
def dashboard_exploratorio(df):

    import plotly.express as px
    from plotly.subplots import make_subplots

    colores = {
        "principal": "#1f3b4d",   # azul oscuro
        "secundario": "#5fa8d3",  # azul claro
        "exito": "#2ca02c",       # verde
        "alerta": "#d62728",      # rojo
        "neutral": "#7f7f7f"      # gris
    }

    #Grafico 1 Caudal promedio por planta
    df_caudal = df.groupby("planta")["caudal_entrada_m3_d"].mean().reset_index()

    fig1 = px.bar(
        df_caudal,
        x="planta",
        y="caudal_entrada_m3_d",
        color="planta",
        color_discrete_sequence=[colores["secundario"]],
        title="1. Caudal Promedio por Planta"
    )

    #Grafico 2 % cumplimiento
    df_cumplimiento = df.groupby("planta")["cumplimiento_norma"].mean().reset_index()

    fig2 = px.bar(
        df_cumplimiento,
        x="planta",
        y="cumplimiento_norma",
        color="cumplimiento_norma",
        color_continuous_scale=["#d62728", "#ffbf00", "#2ca02c"],  # rojo → amarillo → verde
        title="2. % Cumplimiento Norma"
    )

    #Grafico 3 Boxplot eficiencia
    fig3 = px.box(
        df,
        x="planta",
        y="eficiencia_tratamiento",
        color="planta",
        color_discrete_sequence=[
            colores["principal"],
            colores["secundario"],
            "#8ecae6"
        ],
        title="3. Eficiencia de Tratamiento"
    )

    #Grafico 4 Scatter energía vs caudal
    fig4 = px.scatter(
        df,
        x="caudal_entrada_m3_d",
        y="energia_aeracion_kWh",
        color="planta",
        size="lodos_generados_kg_d",
        color_discrete_sequence=[
            colores["principal"],
            colores["secundario"],
            "#8ecae6"
        ],
        title="4. Energía vs Caudal"
    )

    # Layout tipo dashboard
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            "Caudal Promedio",
            "% Cumplimiento",
            "Eficiencia",
            "Energía vs Caudal"
        )
    )

    for trace in fig1.data:
        fig.add_trace(trace, row=1, col=1)

    for trace in fig2.data:
        fig.add_trace(trace, row=1, col=2)

    for trace in fig3.data:
        fig.add_trace(trace, row=2, col=1)

    for trace in fig4.data:
        fig.add_trace(trace, row=2, col=2)

    #Estilo
    fig.update_layout(
        height=800,
        title={
            "text": "Dashboard Operacional - Plantas de Tratamiento",
            "x": 0.5,
            "xanchor": "center",
            "font": {"size": 20, "color": colores["principal"]}
        },
        plot_bgcolor="white",
        paper_bgcolor="white",
     legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.2,
        xanchor="center",
        x=0.5
    ),

    margin=dict(
        t=100,
        r=120,
        b=100
    )
    )
    fig.update_xaxes(showgrid=True, gridcolor="#e6e6e6")
    fig.update_yaxes(showgrid=True, gridcolor="#e6e6e6")

    fig.show()

    return fig


# ------------------------------------------
# 8. Exportar resultados
# ------------------------------------------
def exportar_resultados(df, resumen, indicadores, fig):

    os.makedirs("output", exist_ok=True)

    # Exportar datos
    df.to_csv("output/datos_procesados.csv", index=False)
    resumen.to_excel("output/resumen_planta.xlsx", index=False)

    # Exportar dashboard
    fig.write_html("output/dashboard.html")

    # Exportar indicadores
    with open("output/indicadores.txt", "w") as f:
        for k, v in indicadores.items():
            f.write(f"{k}: {v}\n")

    print(" Resultados exportados en carpeta /output")


# ------------------------------------------
# 8. Ejecución principal
# ------------------------------------------
def main():

    df = cargar_datos()

    evaluar_calidad(df)

    df = limpiar_datos(df)

    guardar_intermedio(df)

    df, indicadores = calcular_indicadores(df)

    resumen = resumen_por_planta(df)

    fig_dashboard = dashboard_exploratorio(df)

    exportar_resultados(df, resumen, indicadores, fig_dashboard)

    print("\n--- Indicadores generales ---")
    print(indicadores)


# ------------------------------------------
# Ejecutar script
# ------------------------------------------
if __name__ == "__main__":
    main()



 Suba archivo .xlsx


Saving dataset_set_A_aguas_residuales.xlsx to dataset_set_A_aguas_residuales (1).xlsx

 ---- Validación de datos ----

--- Valores nulos ---
fecha_registro          0
planta                  0
caudal_entrada_m3_d     0
DBO_entrada_mg_L        0
SST_entrada_mg_L        0
pH_entrada              0
energia_aeracion_kWh    0
lodos_generados_kg_d    0
DBO_salida_mg_L         0
cumplimiento_norma      0
dtype: int64

--- Registros duplicados ---
0

--- Estadísticas generales ---
       caudal_entrada_m3_d  DBO_entrada_mg_L  SST_entrada_mg_L  pH_entrada  \
count           200.000000        200.000000        200.000000   200.00000   
mean           5059.285000        280.145000        232.670000     7.16240   
std            1410.971334         75.566497         64.289398     0.42381   
min            1500.000000         90.000000         70.000000     6.18000   
25%            4193.750000        223.000000        192.500000     6.84750   
50%            5092.500000        284.000000        23

 Resultados exportados en carpeta /output

--- Indicadores generales ---
{'promedio_eficiencia': np.float64(87.09290814394676), 'max_eficiencia': 93.4319526627219, 'min_eficiencia': 80.6923076923077, 'promedio_energia': np.float64(0.2478341078759714)}
